# Kaggriculture Tutorial

Farm your way to market dominance! Players harvest produce and animal products to sell in a dynamic market.

## Game Mechanics 
 
- **Farmer**: takes one action per turn over a season with 30 days and 24 turns per day (720 total).
- **Crops and animals**: each have their own seed cost, time to first yield, and total payout. 
- **Daily care**: plants need watering every day or they turn to weeds, animals need feeding or they escape.
- **Market prices**: move with supply, selling a product pushes its price down, and crops vary in how hard they crash from a glut.
- **Town shops** unlock over the season and steadily buy products, lifting prices over time.
- **Farm hands** can be hired for the day, with increasing costs for each hire per day. 
- **Farm expansion**: start with one quadrant of land and can buy the other three for an escalating fee.
- **Shed**: holds harvested goods but caps at 100 items, anything past that is discarded at end of day.
 - **Win condition**: whoever has the most money in the bank at the end of the season wins.

In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.32.2"

In [ ]:
from kaggle_environments import make

env = make("kaggriculture", debug=True)
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

## Understanding the Observation

Each turn your agent receives an observation with:

- **`player`** — your player id (`0` or `1`).
- **`day`** / **`hour`** — the current in-game day (0-indexed) and turn within that day (0-indexed; there are `turnsPerDay` turns per day).
- **`farms`** — a list of both players' public farm state, indexed by player id. Each farm has:
    - `money` — current bank balance
    - `tiles` — a `boardSize × boardSize` grid indexed `tiles[y][x]`; each cell is `None` (empty), `"LOCKED"` (unowned quadrant), or a dict describing a `PLANT`, `WEED`, `COOP`, or `PASTURE` (see below)
    - `farmer` — `[x, y]` position of your main farmer
    - `hands` — `[x, y]` positions of any hired hands active today
    - `unlocked_quadrants` — subset of `["NW", "NE", "SW", "SE"]`
    - `hires_today` — number of hires already made today (drives the next `HIRE` price)
- **`market`** (shared) — `inventory` and current `prices` per product (`WHEAT, CARROT, TOMATO, STRAWBERRY, MELON, EGG, MILK, WOOL, FERTILIZER`).
- **`town`** (shared) — `unlocked_shops`, the list of shops currently generating per-tick demand.
- **`private`** — your own hidden state (not visible to your opponent):
  - `shed` — counts of every product and animal (`GOOSE`, `COW`, `SHEEP`) stored in your shed
  - `seeds` — seed counts per crop
  - `inventories` — per-unit carried inventories; `[0]` is your main farmer, `[1..]` are today's hired hands in order

Tile dicts come in a few shapes:

- **Plant**: `{kind: "PLANT", crop, planted_day, watered_today, consecutive_unwatered, yield_units, max_lifespan_step, fertilized_until_day}` — `consecutive_unwatered >= 2` turns the tile to a weed at end of day.
- **Weed**: `{kind: "WEED"}` — must be `DIG`-ed before the tile is usable again.
- **Coop / Pasture** (empty): `{kind: "COOP" | "PASTURE"}`.
- **Coop / Pasture** (occupied): adds `animal, placed_day, yield_units, fed_today, consecutive_unfed, cared_today, fertilizer_available, pending_care_bonus`. `consecutive_unfed >= 2` means the animal escapes.

Your agent returns a dict of actions for the farmer, any farm hands, and the market: `{"farmer": 'PASS', "hands": [], "market": []}`


In [ ]:
# Run a quick game to see what the observation looks like
env = make("kaggriculture", debug=True)
env.run(["random", "random"])
from kaggle_environments import make

# 1. Run a slightly longer game to generate meaningful data
env = make("kaggriculture", debug=True)
env.run(["random", "random"])

def analyze_farm_state(step_index=100, player_id=0):
    """A comprehensive diagnostic tool for Kaggriculture observations."""
    # Guard against asking for a step that hasn't happened yet
    if step_index >= len(env.steps):
        step_index = len(env.steps) - 1
        
    obs = env.steps[step_index][0].observation
    my_farm = obs.farms[player_id]
    opp_farm = obs.farms[1 - player_id]
    
    print(f"{'='*60}")
    print(f"🌾 KAGGRICULTURE MATCH STATUS - DAY {obs.day} HOUR {obs.hour} 🌾")
    print(f"{'='*60}")
    
    # --- 1. FINANCIALS & EXPANSION ---
    print(f"💰 [BANK]  You: ${my_farm.money} | Opponent: ${opp_farm.money}")
    print(f"🗺️  [LAND]  You: {len(my_farm.unlocked_quadrants)} quadrants | Opponent: {len(opp_farm.unlocked_quadrants)} quadrants")
    print(f"👷 [LABOR] Hires today: {my_farm.hires_today}")
    
    # --- 2. MARKET TRACKER ---
    print("\n📈 [MARKET PRICES]")
    # Format prices into clean columns
    prices = list(obs.market.prices.items())
    for i in range(0, len(prices), 3):
        chunk = prices[i:i+3]
        print(" | ".join([f"{k:<10}: ${v:>3}" for k, v in chunk]))
        
    # --- 3. PRIVATE INVENTORY ---
    print("\n🏠 [YOUR PRIVATE INVENTORY]")
    shed_items = [f"{k}: {v}" for k, v in obs.private.shed.items() if v > 0]
    seed_items = [f"{k}: {v}" for k, v in obs.private.seeds.items() if v > 0]
    print(f"📦 Shed  ({sum(obs.private.shed.values())}/100 max): {', '.join(shed_items) if shed_items else 'Empty'}")
    print(f"🌱 Seeds: {', '.join(seed_items) if seed_items else 'None'}")
    
    # --- 4. TILE & CROP ANALYSIS ---
    plants, weeds, animals = 0, 0, 0
    for row in my_farm.tiles:
        for tile in row:
            if isinstance(tile, dict):
                if tile.get("kind") == "PLANT": plants += 1
                elif tile.get("kind") == "WEED": weeds += 1
                elif tile.get("kind") in ["COOP", "PASTURE"] and "animal" in tile: animals += 1
                
    print(f"\n🚜 [FARM ACTIVITY]")
    print(f"Active Plants: {plants} | Weeds: {weeds} | Animals: {animals}")
    print(f"{'='*60}\n")

# Run the analyzer on step 100 to see what the farm looks like mid-game
analyze_farm_state(step_index=100, player_id=0)
# Peek at the initial observation
obs = env.steps[1][0].observation  # step 1 = first action step
items = obs.market.prices.keys()
print(f"Player: {obs.player}")
print(f"Player {obs.player}'s Unlocked Farm Areas: {obs.farms[obs.player].unlocked_quadrants}")
for i in items:
    print(f"{i} Price: {obs.market.prices[i]}")

## Agent 1: Melon Maxxer

Our first agent is straightforward:
1. Whenever it runs out of melon seeds and has the cash, buy one more.
2. Walk to the nearest open tile and plant a melon; if it's already standing on a melon plant, water it (or harvest it once it's fully grown).
3. Once melons pile up in the shed, only sell them if the market price is above a threshold, otherwise hold and wait for a better price.
4. Roll the proceeds back into more seeds and repeat.

This demonstrates the core aspects of the game: reading observations, maintaining the farm, and watching the market.   

In [ ]:
import random
from kaggle_environments.envs.kaggriculture.kaggriculture import CROPS

# Agent Configuration
SELL_BATCH_SIZE = 5      # Prevent market crashes by selling in batches
EXPANSION_THRESHOLD = 500 # Minimum cash to keep before buying land/hires
MIN_SELL_PRICE = 50      # Don't sell anything below this price

def _step_toward(fx, fy, tx, ty):
    """Returns the direction to move from (fx, fy) to (tx, ty)."""
    if fx > tx: return "WEST"
    if fx < tx: return "EAST"
    if fy > ty: return "NORTH"
    if fy < ty: return "SOUTH"
    return None

def _get_all_tasks(farm, board_size, available_seeds):
    """Scans the board and identifies all actionable tiles."""
    tasks = []
    for y in range(board_size):
        for x in range(board_size):
            tile = farm["tiles"][y][x]
            if tile == "LOCKED":
                continue
                
            if isinstance(tile, dict) and tile.get("kind") == "PLANT":
                # Harvest if ripe
                if tile.get("yield_units", 0) > 0 and tile.get("planted_day") is not None:
                    tasks.append({"x": x, "y": y, "action": "harvest", "priority": 0})
                # Water if needed
                elif not tile.get("watered_today", True):
                    tasks.append({"x": x, "y": y, "action": "water", "priority": 1})
            elif tile is None and available_seeds:
                tasks.append({"x": x, "y": y, "action": "plant", "priority": 2})
                
    # Sort tasks by priority
    tasks.sort(key=lambda t: t["priority"])
    return tasks

def determine_worker_action(wx, wy, tile, tasks, available_seeds, worker_idx=0):
    """Determines the action for a specific worker (farmer or hand) given their position."""
    
    # If the worker is already standing on a tile that needs attention
    if isinstance(tile, dict) and tile.get("kind") == "PLANT":
        if tile.get("yield_units", 0) > 0:
            return ["HARVEST"]
        elif not tile.get("watered_today", True):
            return ["WATER"]
            
    if tile is None and available_seeds:
        # Pick the most valuable seed we have
        best_seed = max(available_seeds, key=available_seeds.get)
        return ["PLANT", best_seed]
        
    # If no immediate action, move towards the highest priority task
    if tasks:
        # Simple task assignment: workers target different tasks if possible
        target = tasks[worker_idx % len(tasks)]
        step = _step_toward(wx, wy, target["x"], target["y"])
        if step:
            return [step]
            
    return ["PASS"]

def smart_agent(obs):
    farms = obs.get("farms", [])
    player = obs.get("player", 0)
    private = obs.get("private", {}) or {}
    
    if not farms or player >= len(farms):
        return {"farmer": ["PASS"], "hands": [], "market": []}

    farm = farms[player]
    board_size = len(farm["tiles"])
    fx, fy = farm["farmer"]
    farmer_tile = farm["tiles"][fy][fx]
    
    seeds = private.get("seeds", {})
    shed = private.get("shed", {})
    market_prices = (obs.get("market", {}) or {}).get("prices", {})
    money = farm["money"]

    market_actions = []
    
    # --- 1. SMART SELLING (Batched) ---
    for item, qty in shed.items():
        if qty > 0:
            price = market_prices.get(item, 0)
            if price > MIN_SELL_PRICE:
                sell_qty = min(qty, SELL_BATCH_SIZE)
                market_actions.append(["SELL", item, sell_qty])

    # --- 2. DYNAMIC PURCHASING ---
    available_seeds = {k: v for k, v in seeds.items() if v > 0}
    
    # Find the crop with the highest current market price to invest in
    plantable_crops = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]
    best_crop = max(plantable_crops, key=lambda c: market_prices.get(c, 0))
    best_crop_seed_cost = CROPS[best_crop]["seed"]
    
    # Buy seeds if we are running low and have cash
    if sum(available_seeds.values()) < 5 and money > best_crop_seed_cost:
        market_actions.append(["BUY_SEED", best_crop, 1])
        money -= best_crop_seed_cost
        
    # Expand farm if wealthy enough
    if money > EXPANSION_THRESHOLD and len(farm.get("unlocked_quadrants", [])) < 4:
        market_actions.append(["BUY_LAND"])
        money -= 500 # Approximate deduction for logic purposes
        
    # --- 3. LABOR MANAGEMENT ---
    tasks = _get_all_tasks(farm, board_size, available_seeds)
    hands_actions = []
    
    # Hire hands if we have too many tasks
    hires_today = farm.get("hires_today", 0)
    if len(tasks) > 5 and money > EXPANSION_THRESHOLD and hires_today < 2:
        market_actions.append(["HIRE_HAND"])
        
    # --- 4. ACTION DISPATCH ---
    farmer_action = determine_worker_action(fx, fy, farmer_tile, tasks, available_seeds, worker_idx=0)
    
    # Assign actions to any existing hands
    for i, hand_pos in enumerate(farm.get("hands", [])):
        hx, hy = hand_pos
        hand_tile = farm["tiles"][hy][hx]
        hands_actions.append(determine_worker_action(hx, hy, hand_tile, tasks, available_seeds, worker_idx=i+1))

    return {
        "farmer": farmer_action,
        "hands": hands_actions,
        "market": market_actions
    }

In [ ]:
# Test the upgraded smart_agent against the random agent
env = make("kaggriculture", debug=True)
env.run([smart_agent, "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

# Alternative rendering method for Kaggle Notebooks
out = env.render(mode="html")
from IPython.display import HTML
HTML(out)

## Making a submission

You can either submit a main.py, a tar.gz (or zip) with a main.py in it, or submit a notebook with a main.py or submission.tar.gz

There are three ways to subit.
1. using the [Submit Agent](https://www.kaggle.com/competitions/kaggriculture-gdm-internal) button on the homepage and uploading the file
2. using the Kaggle CLI (as described in agents.py in the competition dataset)
3. submitting a notebook with a submission.py or submission.tar.gz

In [ ]:
%%writefile submission.py
import random
from kaggle_environments.envs.kaggriculture.kaggriculture import CROPS

# Agent Configuration
SELL_BATCH_SIZE = 5      
EXPANSION_THRESHOLD = 500 
MIN_SELL_PRICE = 50      

def _step_toward(fx, fy, tx, ty):
    """Returns the direction to move from (fx, fy) to (tx, ty)."""
    if fx > tx: return "WEST"
    if fx < tx: return "EAST"
    if fy > ty: return "NORTH"
    if fy < ty: return "SOUTH"
    return None

def _get_all_tasks(farm, board_size, available_seeds):
    """Scans the board and identifies all actionable tiles."""
    tasks = []
    for y in range(board_size):
        for x in range(board_size):
            tile = farm["tiles"][y][x]
            if tile == "LOCKED":
                continue
                
            if isinstance(tile, dict) and tile.get("kind") == "PLANT":
                if tile.get("yield_units", 0) > 0 and tile.get("planted_day") is not None:
                    tasks.append({"x": x, "y": y, "action": "harvest", "priority": 0})
                elif not tile.get("watered_today", True):
                    tasks.append({"x": x, "y": y, "action": "water", "priority": 1})
            elif tile is None and available_seeds:
                tasks.append({"x": x, "y": y, "action": "plant", "priority": 2})
                
    tasks.sort(key=lambda t: t["priority"])
    return tasks

def determine_worker_action(wx, wy, tile, tasks, available_seeds, worker_idx=0):
    """Determines the action for a specific worker (farmer or hand) given their position."""
    if isinstance(tile, dict) and tile.get("kind") == "PLANT":
        if tile.get("yield_units", 0) > 0:
            return ["HARVEST"]
        elif not tile.get("watered_today", True):
            return ["WATER"]
            
    if tile is None and available_seeds:
        best_seed = max(available_seeds, key=available_seeds.get)
        return ["PLANT", best_seed]
        
    if tasks:
        target = tasks[worker_idx % len(tasks)]
        step = _step_toward(wx, wy, target["x"], target["y"])
        if step:
            return [step]
            
    return ["PASS"]

def smart_agent(obs):
    farms = obs.get("farms", [])
    player = obs.get("player", 0)
    private = obs.get("private", {}) or {}
    
    if not farms or player >= len(farms):
        return {"farmer": ["PASS"], "hands": [], "market": []}

    farm = farms[player]
    board_size = len(farm["tiles"])
    fx, fy = farm["farmer"]
    farmer_tile = farm["tiles"][fy][fx]
    
    seeds = private.get("seeds", {})
    shed = private.get("shed", {})
    market_prices = (obs.get("market", {}) or {}).get("prices", {})
    money = farm["money"]

    market_actions = []
    
    # 1. SMART SELLING (Batched)
    for item, qty in shed.items():
        if qty > 0:
            price = market_prices.get(item, 0)
            if price > MIN_SELL_PRICE:
                sell_qty = min(qty, SELL_BATCH_SIZE)
                market_actions.append(["SELL", item, sell_qty])

    # 2. DYNAMIC PURCHASING
    available_seeds = {k: v for k, v in seeds.items() if v > 0}
    
    plantable_crops = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]
    best_crop = max(plantable_crops, key=lambda c: market_prices.get(c, 0))
    best_crop_seed_cost = CROPS[best_crop]["seed"]
    
    if sum(available_seeds.values()) < 5 and money > best_crop_seed_cost:
        market_actions.append(["BUY_SEED", best_crop, 1])
        money -= best_crop_seed_cost
        
    if money > EXPANSION_THRESHOLD and len(farm.get("unlocked_quadrants", [])) < 4:
        market_actions.append(["BUY_LAND"])
        money -= 500 
        
    # 3. LABOR MANAGEMENT
    tasks = _get_all_tasks(farm, board_size, available_seeds)
    hands_actions = []
    
    hires_today = farm.get("hires_today", 0)
    if len(tasks) > 5 and money > EXPANSION_THRESHOLD and hires_today < 2:
        market_actions.append(["HIRE_HAND"])
        
    # 4. ACTION DISPATCH
    farmer_action = determine_worker_action(fx, fy, farmer_tile, tasks, available_seeds, worker_idx=0)
    
    for i, hand_pos in enumerate(farm.get("hands", [])):
        hx, hy = hand_pos
        hand_tile = farm["tiles"][hy][hx]
        hands_actions.append(determine_worker_action(hx, hy, hand_tile, tasks, available_seeds, worker_idx=i+1))

    return {
        "farmer": farmer_action,
        "hands": hands_actions,
        "market": market_actions
    }

## Submit to competition

Now that we have a main.py, all you need to do is click "Submit to competition" on the right and watch your entry show up on the competition leaderboard! Best of luck!